<a href="https://colab.research.google.com/github/xmegan10/Reddit-Sentiment-Comment-Prediction/blob/main/Reddit%20Comment%20Sentiment%20Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install nltk
%pip install re
%pip install emoji
%pip install contractions
%pip install scikit-learn
%pip install collections
%pip install wordcloud


ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                  

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
import re
import emoji #dealing with emojis
import contractions

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from collections import Counter
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
import os
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /content


In [5]:
os.listdir()

['.config', 'sample_data']

In [6]:
import torch
torch.cuda.is_available()

True

In [1]:
!pip install pyspark -v

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [8]:
from pyspark.sql import SparkSession, Row
spark = SparkSession.builder.appName('Load-jston-to-pyspark-dataframe').getOrCreate()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
df = spark.read.json("/content/drive/MyDrive/Colab Notebooks/Reddit Comment Pred/r_AskReddit_commentsApril.jsonl")

In [13]:
!head -n 10 "/content/drive/MyDrive/Colab Notebooks/Reddit Comment Pred/r_AskReddit_commentsApril.jsonl"

{"_meta":{"retrieved_2nd_on":1775131546},"all_awardings":[],"approved_at_utc":null,"approved_by":null,"archived":false,"associated_award":null,"author":"QuayleDan128","author_flair_background_color":null,"author_flair_css_class":null,"author_flair_richtext":[],"author_flair_template_id":null,"author_flair_text":null,"author_flair_text_color":null,"author_flair_type":"text","author_fullname":"t2_1z8j9sd","author_is_blocked":false,"author_patreon_flair":false,"author_premium":false,"awarders":[],"banned_at_utc":null,"banned_by":null,"body":"probably start pacing like a caged animal lol","can_gild":false,"can_mod_post":false,"collapsed":false,"collapsed_because_crowd_control":null,"collapsed_reason":null,"collapsed_reason_code":null,"comment_type":null,"controversiality":0,"created":1775001601,"created_utc":1775001601,"distinguished":null,"downs":0,"edited":false,"gilded":0,"gildings":{},"id":"odlodpk","is_submitter":false,"likes":null,"link_id":"t3_1s946zf","locked":false,"mod_note":null

In [15]:
result = df.select("*").toPandas()

In [20]:
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833548 entries, 0 to 833547
Data columns (total 73 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   _meta                            810063 non-null  object
 1   all_awardings                    833548 non-null  object
 2   approved_at_utc                  0 non-null       object
 3   approved_by                      0 non-null       object
 4   archived                         833548 non-null  bool  
 5   associated_award                 0 non-null       object
 6   author                           833548 non-null  object
 7   author_cakeday                   2567 non-null    object
 8   author_flair_background_color    23900 non-null   object
 9   author_flair_css_class           0 non-null       object
 10  author_flair_richtext            809899 non-null  object
 11  author_flair_template_id         0 non-null       object
 12  author_flair_tex

In [24]:
result["name"]

,name
0,t1_odlodpk
1,t1_odlodp9
2,t1_odlodrl
3,t1_odlodw2
4,t1_odloe1g
...,...
833543,t1_oewhwvs
833544,t1_oewhwwg
833545,t1_oewhx3a
833546,t1_oewhx3u


In [7]:
df.head()

Row(_meta=Row(retrieved_2nd_on=1775131546), all_awardings=[], approved_at_utc=None, approved_by=None, archived=False, associated_award=None, author='QuayleDan128', author_flair_background_color=None, author_flair_css_class=None, author_flair_richtext=[], author_flair_template_id=None, author_flair_text=None, author_flair_text_color=None, author_flair_type='text', author_fullname='t2_1z8j9sd', author_is_blocked=False, author_patreon_flair=False, author_premium=False, awarders=[], banned_at_utc=None, banned_by=None, body='probably start pacing like a caged animal lol', can_gild=False, can_mod_post=False, collapsed=False, collapsed_because_crowd_control=None, collapsed_reason=None, collapsed_reason_code=None, comment_type=None, controversiality=0, created=1775001601, created_utc=1775001601, distinguished=None, downs=0, edited=False, gilded=0, id='odlodpk', is_submitter=False, likes=None, link_id='t3_1s946zf', locked=False, mod_note=None, mod_reason_by=None, mod_reason_title=None, mod_repo

In [8]:
posts_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Reddit Comment Pred/a_posts_df.csv")
comments_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Reddit Comment Pred/a_comments_df.csv")

posts_df.name = "Ath Posts"
comments_df.name = "Ath Comments"

# EDA

In [ ]:
# head of data set
posts_df.head()

In [ ]:
posts_df.info()

In [ ]:
comments_df.head()

In [ ]:
comments_df.info()

In [ ]:
def create_edadf(*dataframes):
    eda_df = []
    for df in dataframes:
        eda_df.append({
            "name": df.name,
            "num_rows": df.shape[0],
            "num_cols": df.shape[1],
            "contains_null": df.isnull().any(axis = None)
        })
    return pd.DataFrame(eda_df)

p_eda_df = create_edadf(posts_df)
c_eda_df = create_edadf(comments_df)

In [ ]:
p_eda_df

In [ ]:
c_eda_df

In [ ]:
plt.hist(posts_df["post_score"])
plt.xlabel("Post Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
posts_df.info()

In [ ]:
plt.hist(posts_df["num_comments"])
plt.xlabel("Number of Comments")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(comments_df["comment_score"])
plt.xlabel("Comment Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
posts_df = posts_df.copy()
comments_df = comments_df.copy()

# Split Data into Train and Test

In [ ]:
from sklearn.model_selection import train_test_split
ptrain_set, ptest_set = train_test_split(posts_df, test_size = 0.2, random_state = 42)

#get the list of IDs for each split
ptrain_ids = ptrain_set['post_id'].unique()
ptest_ids = ptest_set['post_id'].unique()

ctrain_set = comments_df[comments_df['post_id'].isin(ptrain_ids)]
ctest_set = comments_df[comments_df['post_id'].isin(ptest_ids)]

In [ ]:
ptrain_set.info()

In [ ]:
ctrain_set.info()

In [ ]:
missing_ids = ctrain_set[~ctrain_set['post_id'].isin(ptrain_set['post_id'])]['post_id'].unique()

print(f"Number of unique Post IDs missing from ptrain_setk: {len(missing_ids)}")
print("First 10 missing IDs:", missing_ids[:10])

# Preprocess Text

In [ ]:
ptrain_set["post_string"] = ptrain_set["post_title"].str.cat(ptrain_set["post_text"], sep = " ",na_rep = "")
ptrain_set["post_string"].eq("").sum()

In [ ]:
#add meta data to improve training
#add word counts of post_string as pstring_len

#this may not be necessary in the future if json file includes length of post
ptrain_set["pstring_len"] = ptrain_set["post_string"].str.split().str.len()
ptrain_set.head()



In [ ]:
def preprocess_df(df, text_col = "post_string"):
    #CLEAN TEXT
    processed_texts = []

    #loading stop words and objects from classes
    default_stopwords = set(stopwords.words('english'))
    default_stopwords.add("edit")
    lmtzr = WordNetLemmatizer()

    def clean_text(sentence):
        #return "" if sentence is NaN so no error occurs
        if not isinstance(sentence, str) or sentence.strip() == "":
            return ""

        #convert emojis to text
        sentence = emoji.demojize(sentence)
        #remove https from strings
        sentence = re.sub(r'http\S+','', sentence)
        #remove new lines
        sentence = re.sub(r'\\n|:|_',' ', sentence)
        #expand contractions
        sentence = contractions.fix(sentence)
        #remove punctuations and lowercase
        sentence = re.sub(r'[^a-zA-Z0-9\s]', '', sentence).lower()


        #tokenize words
        tokens = word_tokenize(sentence)

        final_words = []
        for word in tokens: #for each word in the sentence
            fixed_word = contractions.fix(word) #expand contractions
            if fixed_word not in default_stopwords:
                lemma = lmtzr.lemmatize(fixed_word) #lemmatize words
                final_words.append(lemma) #append the singular lemmatized word to the final_words list

        return ' '.join(final_words) #append the list of words to processed_text

    df[text_col] = df[text_col].apply(clean_text)
    return df

In [ ]:
ptrain_set = preprocess_df(ptrain_set, text_col = "post_string")

In [ ]:
ptrain_set["post_string"]

In [ ]:
ptrain_set_test = ptrain_set.copy()

## Pipeline: FeaturePreprocessor

In [ ]:
#write pipeline for preprocess step
from sklearn.base import BaseEstimator, TransformerMixin

class FeaturePreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, variables = None):
        self.variables = variables #placeholder for now; change if need to add specific conditions

        self.stop_words = set(stopwords.words('english'))
        self.stop_words.add("edit")
        self.lmtzr = WordNetLemmatizer()

    def fit(self, X, y = None):
        return self

    def clean_text(self, text):
        if not isinstance(text, str) or text.strip() == "":
            return ""

        #convert emojis to text
        text = emoji.demojize(text)
        #expand contractions
        text = contractions.fix(text)
        #remove https from strings
        text = re.sub(r'http\S+','', text)
        #remove new lines
        text = re.sub(r'\\n|:|_',' ', text)
        #remove punctuations and lowercase
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text).lower()

        #tokenize words
        tokens = word_tokenize(text)
        cleaned_tokens = [self.lmtzr.lemmatize(word) for word in tokens if word not in self.stop_words]

        return cleaned_tokens


    def transform(self, X):
        X = X.copy()

        #combine post_title and post_text into post_string
        X["post_string"] = X["post_title"].str.cat(X["post_text"], sep = " ",na_rep = "")
        #add word counts of post_string as pstring_len
        X["pstring_len"] = X["post_string"].str.split().str.len()

        X["post_string"] = X["post_string"].apply(self.clean_text).str.join(" ")

        return X

In [ ]:
feat_preprocessor = FeaturePreprocessor()
ptrain_set_test = feat_preprocessor.transform(ptrain_set_test)